# **Data Audit & Diagnostic Ingest Pipeline**
This code block executes a comprehensive audit of the raw financial ecosystem. It serves as the "Technical Ground Truth" for the project by identifying the scale, schema, and statistical distribution of all available datasets before they are processed into the modeling pipeline.

In [7]:
import os
from pyspark.sql import functions as F

# 1. Define your data directory
data_path = "/content/drive/MyDrive/CBG_Dataset/"

# 2. List all files in the directory
files = [f for f in os.listdir(data_path) if f.endswith('.csv') or f.endswith('.parquet')]
print(f"Found {len(files)} datasets: {files}\n")

# 3. Iterate through datasets and display statistics
for file_name in files:
    full_path = os.path.join(data_path, file_name)
    print("="*60)
    print(f"DATASET: {file_name}")
    print("="*60)

    # Load based on file type
    if file_name.endswith('.csv'):
        df_temp = spark.read.csv(full_path, header=True, inferSchema=True)
    else:
        df_temp = spark.read.parquet(full_path)

    # Display Row Count and Columns
    print(f"Total Rows: {df_temp.count()}")
    print(f"Schema:")
    df_temp.printSchema()

    # Display Summary Statistics (Mean, StdDev, Min, Max)
    # We select numerical columns only to keep the output readable
    num_cols = [c for c, t in df_temp.dtypes if t in ['int', 'double', 'float', 'bigint']]

    print(f"Summary Statistics for Numerical Columns:")
    # describe() provides count, mean, stddev, min, and max
    df_temp.select(num_cols[:8]).describe().show() # Showing first 8 columns for brevity

    # Check for Null values in key columns
    print("Null Value Count:")
    null_counts = df_temp.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in num_cols[:8]])
    null_counts.show()

    print("\n")

Found 10 datasets: ['HC_POS_CASH_balance.csv', 'HC_application_train.csv', 'HC_bureau.csv', 'HC_bureau_balance.csv', 'HC_credit_card_balance.csv', 'HC_installments_payments.csv', 'HC_sample_submission.csv', 'HomeCredit_columns_description.csv', 'HC_previous_application.csv', 'ABT_Sprint1_Final.parquet']

DATASET: HC_POS_CASH_balance.csv
Total Rows: 10001358
Schema:
root
 |-- SK_ID_PREV: integer (nullable = true)
 |-- SK_ID_CURR: integer (nullable = true)
 |-- MONTHS_BALANCE: integer (nullable = true)
 |-- CNT_INSTALMENT: double (nullable = true)
 |-- CNT_INSTALMENT_FUTURE: double (nullable = true)
 |-- NAME_CONTRACT_STATUS: string (nullable = true)
 |-- SK_DPD: integer (nullable = true)
 |-- SK_DPD_DEF: integer (nullable = true)

Summary Statistics for Numerical Columns:
+-------+------------------+-----------------+-------------------+------------------+---------------------+------------------+-----------------+
|summary|        SK_ID_PREV|       SK_ID_CURR|     MONTHS_BALANCE|    CNT

# Summary of Initial Findings
The initial audit of the CBG-OpenBanking-Canada datasets has provided a high-level view of the data landscape, identifying the fundamental characteristics and technical constraints that will shape the project's development. These findings serve as the baseline for all future modeling and engineering efforts.

**1. Data Landscape & Distribution**
The primary dataset exhibits a significant "Class Imbalance," which is standard in risk-based financial modeling.

Observation: The majority of the population represents successful repayments, while a small minority represents the event of interest (risk).

Impact: This skew requires specific algorithmic handling, such as class weighting or synthetic sampling, to ensure the model does not become biased toward the majority group.

**2. Feature Variance and Scale**
An analysis of the numerical features revealed a wide disparity in "Magnitudes" across different data points.

Observation: Financial values (like income or credit limits) exist on a much larger scale than demographic values (like age or duration of employment).

Impact: Without normalization or standard scaling, traditional machine learning models may inadvertently prioritize features with larger raw numbers, regardless of their actual predictive value.

**3. Behavioral Signal Potential**
The transactional and installment datasets provide a "High-Resolution" view of user behavior that is missing from traditional static applications.

Observation: The frequency and timing of interactions within the financial ecosystem provide thousands of potential data points.

Impact: This confirms that the raw data contains enough granular detail to support the creation of alternative credit metrics, moving the project's focus toward "Alternative Data" sources.

**4. Technical Performance Baseline**
The initial diagnostic runs provided a baseline for model accuracy and predictive reliability.

Observation: The first-pass evaluation metric (ROC-AUC) indicated that the raw features require further refinement to establish a strong predictive relationship with the target variable.

Impact: This baseline acts as a "Ground Truth" for measuring the improvement gained from subsequent feature engineering and advanced modeling techniques.

**Conclusion of Initial Audit**
The foundational analysis confirms that while the raw data is complex and imbalanced, it possesses the necessary depth to build a non-traditional risk assessment tool. The focus now transitions from understanding the raw data to "Engineering the Signal" through advanced mathematical transformations and ensemble modeling.